In [1]:
import pandas as pd
import numpy as np

from decimal import Decimal, ROUND_HALF_UP
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Labs"
)

xpt_file = DATA_DIR / "TCHOL_G.xpt"
csv_file = DATA_DIR / "TCHOL_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched source values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2011-2012 TCHOL_G XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "LBXTC",
    "LBDTCSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TCHOL_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 7821:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 7,821 rows x 3 columns."
)


# ============================================================
# 4. DEFINE VARIABLE TYPES
# ============================================================

#
# SEQN
#     Respondent identifier
#     whole-number
#
# LBXTC
#     Total cholesterol, mg/dL
#     released as whole-number values
#
# LBDTCSI
#     Total cholesterol, mmol/L
#     genuine decimal laboratory value
#

INTEGER_COLUMNS = [
    "SEQN",
    "LBXTC"
]


DECIMAL_COLUMNS = [
    "LBDTCSI"
]


print("\n--- VARIABLE TYPE PLAN ---")

print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached TCHOL_G.xpt:
#
#     tiny artifacts = 0
#
# Therefore NO zero restoration is needed.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros:",
    true_zero_total
)

print(
    "Tiny XPORT values:",
    tiny_total
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT values "
        "found in TCHOL_G. Do not automatically "
        "replace them with zero."
    )


if true_zero_total != 0:

    raise ValueError(
        "STOP: Unexpected numeric zeros "
        "found in TCHOL_G."
    )


print(
    "PASS: No tiny XPORT artifacts."
)

print(
    "PASS: No numeric zeros."
)


# ============================================================
# 6. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# Must occur BEFORE Int64 conversion.
#
# Expected:
#
#     SEQN       whole-number
#     LBXTC      whole-number
#     LBDTCSI    genuine decimal
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns != ["LBDTCSI"]:

    raise ValueError(
        "Unexpected decimal-variable pattern.\n"
        "Expected: ['LBDTCSI']\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: LBDTCSI is the only genuine "
    "decimal variable."
)


# ============================================================
# 7. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. FORMAT LBXTC WITHOUT .0
# ============================================================

#
# LBXTC is released as whole-number mg/dL.
#
# Example:
#
#     168.0 -> 168
#
# NO .round() is used.
#

values = (
    df["LBXTC"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.all(
    np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "LBXTC unexpectedly contains "
        "fractional values."
    )


df["LBXTC"] = (
    df["LBXTC"]
    .astype("Int64")
)


print(
    "PASS: LBXTC converted to Int64 "
    "without rounding."
)


# ============================================================
# 9. PRESERVE LBDTCSI EXACTLY
# ============================================================

#
# LBDTCSI is genuine decimal mmol/L.
#
# DO NOT:
#
#     round()
#     astype(Int64)
#     use float_format
#

if not np.allclose(
    source_df["LBDTCSI"]
    .to_numpy(
        dtype=float
    ),
    df["LBDTCSI"]
    .to_numpy(
        dtype=float
    ),
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBDTCSI changed unexpectedly."
    )


print(
    "PASS: LBDTCSI preserved exactly."
)


# ============================================================
# 10. CDC CHECK - LBXTC
# ============================================================

#
# CDC:
#
# LBXTC = Total Cholesterol (mg/dL)
#
# Range:
#
#     59 - 523
#
# Valid:
#
#     6,988
#
# Missing:
#
#     833
#

s = df["LBXTC"]


print("\n--- LBXTC ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    s.min() == 59
    and
    s.max() == 523
    and
    int(
        s.notna()
        .sum()
    ) == 6988
    and
    int(
        s.isna()
        .sum()
    ) == 833
):

    raise ValueError(
        "LBXTC differs from CDC."
    )


print(
    "PASS: LBXTC matches CDC."
)


# ============================================================
# 11. CDC CHECK - LBDTCSI
# ============================================================

#
# CDC:
#
# LBDTCSI = Total Cholesterol (mmol/L)
#
# Range:
#
#     1.53 - 13.52
#
# Valid:
#
#     6,988
#
# Missing:
#
#     833
#

s = df["LBDTCSI"]


print("\n--- LBDTCSI ---")

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Non-missing:",
    int(
        s.notna()
        .sum()
    )
)

print(
    "Missing:",
    int(
        s.isna()
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        1.53,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        13.52,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    ) == 6988
    and
    int(
        s.isna()
        .sum()
    ) == 833
):

    raise ValueError(
        "LBDTCSI differs from CDC."
    )


print(
    "PASS: LBDTCSI matches CDC."
)


# ============================================================
# 12. VERIFY CDC SI CONVERSION
# ============================================================

#
# CDC formula:
#
#     LBDTCSI
#
#        =
#
#     LBXTC × 0.02586
#
#
# Released SI values use two decimal places.
#
# Use Decimal + ROUND_HALF_UP rather than Python round()
# because Python uses banker's rounding.
#

def convert_total_cholesterol_to_si(
    mg_dl
):

    converted = (
        Decimal(
            str(int(mg_dl))
        )
        *
        Decimal("0.02586")
    )


    return float(
        converted.quantize(
            Decimal("0.01"),
            rounding=ROUND_HALF_UP
        )
    )


paired = (
    df[
        [
            "LBXTC",
            "LBDTCSI"
        ]
    ]
    .dropna()
)


expected_si = (
    paired["LBXTC"]
    .apply(
        convert_total_cholesterol_to_si
    )
    .to_numpy(
        dtype=float
    )
)


actual_si = (
    paired["LBDTCSI"]
    .to_numpy(
        dtype=float
    )
)


si_mismatches = int(
    np.sum(
        expected_si
        !=
        actual_si
    )
)


print("\n--- CDC SI CONVERSION CHECK ---")

print(
    "Pairs checked:",
    f"{len(paired):,}"
)

print(
    "Conversion mismatches:",
    si_mismatches
)


if si_mismatches != 0:

    raise ValueError(
        "LBDTCSI does not match "
        "LBXTC × 0.02586."
    )


print(
    "PASS: All 6,988 LBDTCSI values "
    "match CDC conversion exactly."
)


# ============================================================
# 13. CHECK LEGITIMATE WHOLE-LOOKING DECIMAL VALUES
# ============================================================

#
# Some SI values happen to be numerically whole:
#
#     4.0
#     5.0
#
# They are still genuine decimal laboratory results.
#
# Attached TCHOL_G contains 33 such values.
#

si_nonmissing = (
    df["LBDTCSI"]
    .dropna()
)


whole_looking_mask = np.isclose(
    si_nonmissing,
    np.round(
        si_nonmissing
    ),
    rtol=0,
    atol=1e-12
)


whole_looking_count = int(
    whole_looking_mask.sum()
)


print(
    "\nLBDTCSI whole-number-looking decimals:",
    whole_looking_count
)


if whole_looking_count != 33:

    raise ValueError(
        "Unexpected count of whole-looking "
        "LBDTCSI values."
    )


print(
    "PASS: 33 legitimate decimal values "
    "such as 4.0 / 5.0 are preserved."
)


# ============================================================
# 14. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# TCHOL_G requires NO numeric correction.
#
# Therefore all numeric values must remain exactly
# equal to the source XPT.
#
# Only CSV formatting changes:
#
#     SEQN
#     62161.0 -> 62161
#
#     LBXTC
#     168.0 -> 168
#
# LBDTCSI stays decimal.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 3 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     float_format=
#
# NO:
#
#     rounding
#
# This preserves genuine LBDTCSI decimal values naturally.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 16. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 17. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 CSV variables round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 18. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 19. READ RAW CSV AS TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 20. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 21. .0 CHECK - WHOLE-NUMBER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# Do NOT check LBDTCSI for ".0" as an error.
#
# It is a genuine decimal laboratory variable.
#
# Only:
#
#     SEQN
#     LBXTC
#
# should have unnecessary '.0' removed.
#

print("\n--- INTEGER .0 FORMAT CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 in "
    "SEQN or LBXTC."
)


# ============================================================
# 22. LBDTCSI .0 VALUES ARE LEGITIMATE
# ============================================================

#
# A value such as:
#
#     4.0 mmol/L
#
# is a legitimate laboratory result.
#
# It should NOT be converted to integer 4 merely
# because it happens to have no fractional remainder.
#

si_dot_zero_count = int(
    (
        raw_csv_df["LBDTCSI"].ne("")
        &
        raw_csv_df["LBDTCSI"]
        .str.endswith(".0")
    )
    .sum()
)


print("\n--- LBDTCSI DECIMAL FORMAT CHECK ---")

print(
    "Legitimate LBDTCSI values ending in .0:",
    si_dot_zero_count
)


if si_dot_zero_count != 33:

    raise ValueError(
        "Unexpected LBDTCSI '.0' representation count."
    )


print(
    "PASS: Legitimate LBDTCSI '.0' values preserved."
)


# ============================================================
# 23. EXACT DECIMAL FIDELITY
# ============================================================

expected_values = (
    source_df["LBDTCSI"]
    .to_numpy(
        dtype=float
    )
)


exported_values = (
    roundtrip_df["LBDTCSI"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_values,
    exported_values,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "LBDTCSI decimal fidelity failed."
    )


print(
    "PASS: LBDTCSI decimal values "
    "preserved exactly."
)


# ============================================================
# 24. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 25. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "TCHOL_G CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Whole-number variables:",
    INTEGER_COLUMNS
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "LBXTC valid results:",
    int(
        df["LBXTC"]
        .notna()
        .sum()
    )
)


print(
    "LBDTCSI valid results:",
    int(
        df["LBDTCSI"]
        .notna()
        .sum()
    )
)


print(
    "SI conversion mismatches:",
    si_mismatches
)


print(
    "Legitimate LBDTCSI '.0' values:",
    si_dot_zero_count
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0 in integer fields: "
    "0 expected"
)


print(
    "Lab decimal values altered: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 TCHOL_G XPT -> CSV
Rows: 7,821
Columns: 3
PASS: Structure = 7,821 rows x 3 columns.

--- VARIABLE TYPE PLAN ---
Whole-number variables: ['SEQN', 'LBXTC']
Genuine decimal variables: ['LBDTCSI']

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT values: 0
PASS: No tiny XPORT artifacts.
PASS: No numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
LBXTC: 0 fractional observations
LBDTCSI: 6,955 fractional observations
PASS: LBDTCSI is the only genuine decimal variable.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.
PASS: LBXTC converted to Int64 without rounding.
PASS: LBDTCSI preserved exactly.

--- LBXTC ---
Minimum: 59
Maximum: 523
Non-missing: 6988
Missing: 833
PASS: LBXTC matches CDC.

--- LBDTCSI ---
Minimum: 1.53
Maximum: 13.52
Non-missing: 6988
Missing: 833
PASS: LBDTCSI matches CDC.

--- CDC SI CONVERSION CHECK ---
Pairs checked: 6,988
Conversion mismatches: 0
PASS: All 6,988